# Summary_Day5.ipynb  
## 경사하강법 구현 · 예측 함수 · Layer 함수 · Gradient 안정성 · BatchNorm

이번 5강은 “PyTorch로 모델을 만든다는 것이 실제 코드에서는 어떤 모양인가”를 정리하는 강의다.

4강에서 `W`, `B`를 직접 만들고 경사하강법을 구현했다면,  
5강에서는 그 구조가 `nn.Linear`, `nn.ReLU`, `nn.Sequential`, `optimizer`, `DataLoader`, `BatchNorm`으로 확장된다.

강의 흐름을 한 줄로 정리하면 다음이다.

```text
예측 함수 = Layer 함수들의 합성 함수
학습 = Layer 내부 parameter를 손실이 줄어드는 방향으로 수정하는 과정
```

이번 강의에서 꼭 잡아야 하는 포인트는 다음이다.

1. 신경망 그림과 PyTorch 프로그램 모델의 차이를 이해한다.
2. Layer 함수, Parameter, Model, Learning의 의미를 정리한다.
3. `nn.Linear(784, 128)`처럼 입력/출력 차원을 맞추는 이유를 이해한다.
4. `nn.Sequential()`로 여러 Layer를 하나의 예측 함수처럼 묶는다.
5. PyTorch 학습의 3대 요소인 `net`, `criterion`, `optimizer`를 정리한다.
6. 학습 4단계인 예측, 손실, 경사, 업데이트를 코드 흐름으로 익힌다.
7. 활성화 함수가 없으면 깊게 쌓아도 선형 모델과 같다는 점을 실험한다.
8. `argmax()`, `item()`, `max()` 차이를 정리한다.
9. Gradient Hook으로 각 Layer의 gradient 흐름을 관찰한다.
10. Learning Rate Scheduler로 학습률을 조절한다.
11. He 초기화와 Xavier 초기화의 차이를 정리한다.
12. Mini-batch와 Batch size가 학습에 미치는 영향을 확인한다.
13. BatchNorm이 gradient 안정성과 학습 속도에 주는 효과를 정리한다.
14. `model.train()`과 `model.eval()`의 차이를 이해한다.
15. BatchNorm과 LayerNorm의 사용 상황을 구분한다.

> 필기 포인트:  
> 이번 강의는 “신경망 그림”보다 “텐서가 함수들을 어떤 순서로 통과하는가”를 보는 것이 핵심이다.  
> PyTorch에서는 모델을 그림으로 생각하기보다, 입력 Tensor가 여러 함수들을 지나 출력 Tensor가 되는 흐름으로 보면 된다.

## 1. 라이브러리 준비

이번 실습에서는 NumPy, Matplotlib, PyTorch, scikit-learn을 사용한다.

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
```

- `torch`: Tensor와 자동 미분을 사용할 때 필요하다.
- `nn`: `Linear`, `ReLU`, `BatchNorm`, `Loss` 같은 신경망 구성 요소를 제공한다.
- `optim`: SGD, Adam, AdamW 같은 Optimizer를 제공한다.
- `TensorDataset`: 입력 Tensor와 정답 Tensor를 묶는다.
- `DataLoader`: 데이터를 mini-batch 단위로 꺼낸다.

> 실습 메모:  
> 원본 5강 코드에는 MNIST 다운로드와 torchviz 시각화 코드가 있다.  
> 이 Summary는 인터넷 없이도 실행되도록 더미 데이터와 scikit-learn 합성 데이터를 사용한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

%matplotlib inline

np.set_printoptions(suppress=True, precision=4)

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)
print("PyTorch version:", torch.__version__)

## 2. 핵심 용어 정리

강의 자료에서 가장 중요한 개념은 다음이다.

```text
머신러닝 모델 = Layer 함수 구조와 Parameter 값의 유기적인 결합
```

| 용어 | 의미 |
|---|---|
| Layer 함수 | Tensor를 입력받아 Tensor를 출력하는 함수 |
| Parameter | Layer 안에 저장되어 학습으로 바뀌는 값 |
| Model | 여러 Layer 함수를 조합한 하나의 큰 합성 함수 |
| Learning | Parameter를 정답에 가까워지도록 수정하는 과정 |

> 헷갈림 포인트:  
> 신경망 그림의 “층”은 데이터가 놓이는 위치처럼 보이지만,  
> PyTorch 코드에서 Layer는 실제로 Tensor를 변환하는 함수에 가깝다.

In [ ]:
terms = {
    "Layer 함수": "Tensor를 입력받아 Tensor를 출력하는 함수",
    "Parameter": "Layer 내부에 저장되어 학습으로 조정되는 값",
    "Model": "Layer 함수들을 조합한 큰 합성 함수",
    "Learning": "정답에 가까워지도록 Parameter를 수정하는 과정"
}

for key, value in terms.items():
    print(f"{key}: {value}")

## 3. 손글씨 숫자 분류 모델의 Layer 함수 만들기

강의 예시는 28×28 이미지를 펼쳐서 784개의 숫자로 만든 뒤, 0~9 숫자 중 하나로 분류하는 구조다.

```text
Image 28×28
→ Vector 784
→ Hidden 128
→ Class 10
```

### 함수 사용법: `nn.Linear()`

```python
nn.Linear(in_features, out_features)
```

- `in_features`: 입력 feature 개수다.
- `out_features`: 출력 feature 개수다.
- 내부에는 학습되는 `weight`와 `bias`가 있다.

> 중요한 점:  
> 앞 Layer의 출력 수와 다음 Layer의 입력 수가 반드시 맞아야 한다.  
> `Linear(784, 128)` 다음에는 `Linear(128, 10)`처럼 128이 이어져야 한다.

In [ ]:
l1 = nn.Linear(784, 128)
l2 = nn.Linear(128, 10)
relu = nn.ReLU(inplace=True)

print(l1)
print(l2)
print(relu)

### 함수 사용법: `nn.ReLU(inplace=True)`

```python
nn.ReLU(inplace=True)
```

- 음수는 0으로 바꾼다.
- 양수는 그대로 통과시킨다.
- `inplace=True`는 가능한 경우 원본 Tensor를 직접 수정한다는 뜻이다.

> 주의:  
> `inplace=True`는 메모리를 아낄 수 있지만, 계산 그래프가 복잡한 경우 주의가 필요하다.  
> 초보자 입장에서는 `inplace=False` 기본값도 충분히 자주 쓴다.

## 4. 더미 입력 데이터로 Forward 흐름 확인하기

실제 MNIST를 다운로드하지 않고, 같은 shape의 더미 데이터를 만든다.

### 함수 사용법: `torch.randn()`

```python
torch.randn(100, 784)
```

- 평균 0, 표준편차 1의 정규분포 난수 Tensor를 만든다.
- 여기서는 100개의 이미지 데이터가 있고, 각 데이터가 784개 feature를 가진다고 가정한다.

In [ ]:
inputs = torch.randn(100, 784)

m1 = l1(inputs)
m2 = relu(m1)
outputs = l2(m2)

print("inputs shape:", inputs.shape)
print("m1 shape:", m1.shape)
print("m2 shape:", m2.shape)
print("outputs shape:", outputs.shape)

출력 shape 흐름:

```text
[100, 784]
→ Linear(784, 128)
→ [100, 128]
→ ReLU
→ [100, 128]
→ Linear(128, 10)
→ [100, 10]
```

> 필기 포인트:  
> batch 크기 100은 그대로 유지되고, feature 차원만 Layer를 지나며 바뀐다.

## 5. nn.Sequential로 하나의 예측 함수 만들기

위에서 직접 `l1 → relu → l2` 순서로 실행했다.

`nn.Sequential()`을 쓰면 이 순서를 하나의 모델처럼 묶을 수 있다.

### 함수 사용법

```python
net = nn.Sequential(l1, relu, l2)
outputs = net(inputs)
```

- 괄호 안에 Layer를 순서대로 넣는다.
- 입력 Tensor가 첫 Layer부터 마지막 Layer까지 자동으로 지나간다.

In [ ]:
net_digit = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

outputs2 = net_digit(inputs)

print(net_digit)
print("outputs2 shape:", outputs2.shape)

> 기억할 점:  
> `nn.Sequential`은 Layer를 순서대로 쌓는 가장 간단한 방식이다.  
> 복잡한 모델에서는 `class MyModel(nn.Module)`로 직접 `forward()`를 정의한다.

## 6. PyTorch 학습의 3대 핵심 함수

강의 자료에서 강조한 세 가지 요소다.

```text
1. 예측 함수 net
2. 손실 함수 criterion
3. 최적화 함수 optimizer
```

이 세 가지가 없으면 학습이 시작되지 않는다.

| 요소 | 역할 |
|---|---|
| `net` | 입력을 받아 예측값을 만든다 |
| `criterion` | 예측값과 정답을 비교해 손실을 계산한다 |
| `optimizer` | gradient를 바탕으로 parameter를 수정한다 |

In [ ]:
net = nn.Sequential(
    nn.Linear(1, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)

criterion = nn.MSELoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

print("net:", net)
print("criterion:", criterion)
print("optimizer:", optimizer)

### 함수 사용법 정리

```python
nn.MSELoss()
```

- 회귀 문제에서 예측값과 정답의 평균 제곱 오차를 계산한다.

```python
optim.SGD(net.parameters(), lr=0.01)
```

- `net.parameters()`는 모델 안의 모든 weight와 bias를 Optimizer에게 넘긴다.
- `lr`은 learning rate다.

## 7. 학습의 4단계 순환

PyTorch 학습 루프는 거의 항상 다음 순서다.

```text
1. outputs = net(inputs)
2. loss = criterion(outputs, labels)
3. loss.backward()
4. optimizer.step()
```

그리고 새 반복 전에 gradient를 비운다.

```python
optimizer.zero_grad()
```

> 시험 포인트:  
> `zero_grad()` → `forward` → `loss` → `backward` → `step` 순서를 기억하면 된다.

In [ ]:
x_demo = torch.tensor([[1.0], [2.0], [3.0]])
y_demo = torch.tensor([[2.0], [4.0], [6.0]])

optimizer.zero_grad()

outputs_demo = net(x_demo)
loss_demo = criterion(outputs_demo, y_demo)

loss_demo.backward()
optimizer.step()

print("outputs:")
print(outputs_demo)
print("loss:", loss_demo.item())

## 8. 2차 함수 데이터 만들기

활성화 함수가 왜 필요한지 보기 위해 2차 함수 형태 데이터를 만든다.

```text
y = x² + noise
```

선형 모델은 이 곡선을 제대로 따라가기 어렵다.

In [ ]:
np.random.seed(123)

x = np.random.randn(100, 1) * 2.5
y = x ** 2 + np.random.randn(100, 1) * 0.8

x_train = x[:50, :]
y_train = y[:50, :]

x_test = x[50:, :]
y_test = y[50:, :]

inputs = torch.tensor(x_train).float()
labels = torch.tensor(y_train).float()

inputs_test = torch.tensor(x_test).float()
labels_test = torch.tensor(y_test).float()

print("inputs:", inputs.shape)
print("labels:", labels.shape)
print("inputs_test:", inputs_test.shape)
print("labels_test:", labels_test.shape)

### 함수 사용법: `torch.tensor(...).float()`

```python
torch.tensor(x_train).float()
```

- NumPy 배열을 PyTorch Tensor로 바꾼다.
- `.float()`는 float32 타입으로 바꾼다.
- 모델 학습용 입력과 정답은 보통 float Tensor로 만든다.

In [ ]:
plt.scatter(inputs, labels, label="train")
plt.scatter(inputs_test, labels_test, marker="x", label="test")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Quadratic Data")
plt.legend()
plt.show()

그래프 해석:

- 점들이 직선이 아니라 U자 형태에 가깝다.
- 이 데이터는 단순 선형회귀로는 잘 맞추기 어렵다.
- 활성화 함수가 들어간 딥러닝 모델이 필요한 이유를 확인하기 좋은 예제다.

## 9. 모델 1: 선형 회귀 모델

첫 번째 모델은 가장 단순한 선형 모델이다.

```text
Linear(1 → 1)
```

### 클래스 구조

```python
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(1, 1)

    def forward(self, x):
        return self.l1(x)
```

- `__init__`: Layer를 준비한다.
- `forward`: 입력이 Layer를 통과하는 순서를 정의한다.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(1, 1)

    def forward(self, x):
        x1 = self.l1(x)
        return x1

net1 = Net()

print(net1)

> 기억할 점:  
> `net1`은 직선 하나만 만들 수 있다.  
> 2차 함수처럼 휘어진 데이터를 따라가기는 어렵다.

## 10. 모델 2: 활성화 함수 없는 깊은 모델

두 번째 모델은 Linear Layer를 3개 쌓는다.

하지만 활성화 함수가 없다.

```text
Linear → Linear → Linear
```

겉으로는 깊어 보이지만, 수학적으로는 결국 하나의 선형 함수와 같다.

In [ ]:
class Net2(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(1, 10)
        self.l2 = nn.Linear(10, 10)
        self.l3 = nn.Linear(10, 1)

    def forward(self, x):
        x1 = self.l1(x)
        x2 = self.l2(x1)
        x3 = self.l3(x2)
        return x3

net2 = Net2()

print(net2)

> 시험 포인트:  
> 선형 함수의 합성은 여전히 선형이다.  
> 그래서 활성화 함수 없이 Layer만 많이 쌓는 것은 깊은 신경망의 의미가 거의 없다.

## 11. 모델 3: ReLU가 있는 딥러닝 모델

세 번째 모델은 Linear 사이에 ReLU를 넣는다.

```text
Linear → ReLU → Linear → ReLU → Linear
```

활성화 함수가 들어가면서 비선형 패턴을 학습할 수 있다.

In [ ]:
class Net3(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(1, 10)
        self.l2 = nn.Linear(10, 10)
        self.l3 = nn.Linear(10, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x1 = self.relu(self.l1(x))
        x2 = self.relu(self.l2(x1))
        x3 = self.l3(x2)
        return x3

net3 = Net3()

print(net3)

> 필기 포인트:  
> ReLU는 단순한 직선 조각들을 조합해 복잡한 곡선을 만들 수 있게 도와준다.

## 12. 학습 함수 만들기

세 모델을 같은 방식으로 학습시키기 위해 함수로 묶는다.

### 함수 사용법

```python
train_regression_model(model, inputs, labels, num_epochs=1000, lr=0.01)
```

- `model`: 학습할 PyTorch 모델이다.
- `inputs`: 훈련 입력 Tensor다.
- `labels`: 훈련 정답 Tensor다.
- `num_epochs`: 반복 횟수다.
- `lr`: learning rate다.

반환값은 학습된 모델과 loss 기록이다.

In [ ]:
def train_regression_model(model, inputs, labels, num_epochs=1000, lr=0.01):
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = []

    for epoch in range(num_epochs):
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            history.append([epoch, loss.item()])

    return model, np.array(history)

함수 내부 흐름:

```text
optimizer.zero_grad()
→ outputs = model(inputs)
→ loss = criterion(outputs, labels)
→ loss.backward()
→ optimizer.step()
```

이 흐름이 PyTorch 학습의 기본 패턴이다.

## 13. 세 모델 학습하기

세 모델을 같은 데이터로 학습한다.

In [ ]:
torch.manual_seed(42)

net1 = Net()
net2 = Net2()
net3 = Net3()

net1, hist1 = train_regression_model(net1, inputs, labels)
net2, hist2 = train_regression_model(net2, inputs, labels)
net3, hist3 = train_regression_model(net3, inputs, labels)

print("net1 final loss:", hist1[-1, 1])
print("net2 final loss:", hist2[-1, 1])
print("net3 final loss:", hist3[-1, 1])

결과 해석:

- `net1`: 직선만 가능하다.
- `net2`: Layer는 많지만 활성화 함수가 없어 여전히 선형에 가깝다.
- `net3`: ReLU 덕분에 곡선 형태를 더 잘 따라갈 수 있다.

In [ ]:
plt.plot(hist1[:, 0], hist1[:, 1], label="Linear")
plt.plot(hist2[:, 0], hist2[:, 1], label="Deep without activation")
plt.plot(hist3[:, 0], hist3[:, 1], label="Deep with ReLU")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

그래프 해석:

- Loss가 낮을수록 훈련 데이터에 더 잘 맞는다.
- 활성화 함수가 있는 모델이 더 낮은 손실로 갈 가능성이 높다.
- 단, 모델이 너무 복잡하면 과적합 가능성도 같이 생각해야 한다.

## 14. 세 모델의 예측 결과 비교

테스트 데이터에 대한 예측을 그래프로 확인한다.

In [ ]:
x_line = torch.linspace(inputs_test.min(), inputs_test.max(), 200).view(-1, 1)

with torch.no_grad():
    y1_line = net1(x_line)
    y2_line = net2(x_line)
    y3_line = net3(x_line)

plt.scatter(inputs_test, labels_test, marker="x", label="test data")
plt.plot(x_line, y1_line, label="Linear")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Model 1: Linear Regression")
plt.legend()
plt.show()

In [ ]:
plt.scatter(inputs_test, labels_test, marker="x", label="test data")
plt.plot(x_line, y2_line, label="Deep without activation")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Model 2: Deep without Activation")
plt.legend()
plt.show()

In [ ]:
plt.scatter(inputs_test, labels_test, marker="x", label="test data")
plt.plot(x_line, y3_line, label="Deep with ReLU")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Model 3: Deep with ReLU")
plt.legend()
plt.show()

그래프 해석:

- 선형 모델은 직선 형태만 만든다.
- 활성화 함수 없는 깊은 모델도 여전히 직선에 가깝다.
- ReLU가 있는 모델은 곡선 형태를 어느 정도 따라간다.

> 핵심 문장:  
> 활성화 함수가 있어야 층을 깊게 쌓는 의미가 생긴다.

## 15. argmax, item, max 차이

5강 보충 코드에서 `argmax`, `item`, `max` 차이를 정리했다.

### 함수 사용법: `argmax()`

```python
tensor.argmax()
tensor.argmax(dim=1)
```

- 가장 큰 값의 위치 index를 반환한다.
- 분류 모델에서 예측 class를 고를 때 자주 쓴다.

### 함수 사용법: `item()`

```python
tensor.item()
```

- 원소 하나짜리 Tensor에서 Python 숫자를 꺼낸다.

### 함수 사용법: `max()`

```python
tensor.max()
```

- 가장 큰 값 자체를 반환한다.

In [ ]:
scores = torch.tensor([10, 25, 7, 35])

print("scores:", scores)
print("argmax:", scores.argmax())
print("argmax item:", scores.argmax().item())
print("max:", scores.max())

In [ ]:
out = torch.tensor([
    [0.1, 2.3, 0.5, 1.1],
    [3.2, 0.4, 1.8, 0.9],
    [0.2, 1.1, 4.0, 0.3]
])

print("out shape:", out.shape)
print("전체 argmax:", out.argmax())
print("행별 argmax:", out.argmax(1))
print("전체 max:", out.max())

해석:

- `out.argmax()`는 전체 Tensor를 펼쳐서 가장 큰 값의 위치를 본다.
- `out.argmax(1)`은 각 행에서 가장 큰 값의 index를 찾는다.
- 분류에서는 보통 batch마다 class를 골라야 하므로 `out.argmax(1)`을 자주 쓴다.

## 16. Gradient Hook 개념

Gradient Hook은 역전파 중 gradient를 관찰하거나 수정할 수 있는 기능이다.

강의에서는 CCTV처럼 각 층에 설치해서 gradient가 어떻게 흐르는지 관찰하는 기능으로 설명했다.

### 함수 사용법

```python
param.register_hook(hook_function)
```

- 특정 Tensor의 gradient가 계산될 때 hook 함수가 실행된다.
- gradient 평균, 표준편차, 크기를 기록할 수 있다.

> 주의:  
> Hook은 디버깅이나 분석용으로 유용하지만, 초보자는 먼저 학습 루프를 정확히 이해하는 것이 우선이다.

In [ ]:
simple_layer = nn.Linear(3, 1)

def gradient_hook(grad):
    print("gradient mean:", grad.mean().item())
    print("gradient std:", grad.std().item())
    return grad

hook_handle = simple_layer.weight.register_hook(gradient_hook)

sample_x = torch.randn(5, 3)
sample_y = torch.randn(5, 1)

sample_out = simple_layer(sample_x)
sample_loss = nn.MSELoss()(sample_out, sample_y)

sample_loss.backward()

hook_handle.remove()

출력 해석:

- `gradient mean`은 gradient 평균이다.
- `gradient std`는 gradient가 얼마나 퍼져 있는지 보여준다.
- 너무 작으면 기울기 소실, 너무 크면 기울기 폭발을 의심할 수 있다.

## 17. 분류 데이터 준비

Gradient 분포, Scheduler, BatchNorm 실습을 위해 합성 분류 데이터를 만든다.

### 함수 사용법: `make_classification()`

```python
make_classification(n_samples=1000, n_features=20, n_classes=2)
```

- 이진 분류용 가짜 데이터를 만든다.
- `n_features=20`이면 입력 feature가 20개다.

In [ ]:
X_cls, y_cls = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_cls,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train)
X_test_t = torch.FloatTensor(X_test)
y_train_t = torch.LongTensor(y_train)
y_test_t = torch.LongTensor(y_test)

print("X_train_t:", X_train_t.shape)
print("y_train_t:", y_train_t.shape)

### 함수 사용법: `StandardScaler`

```python
scaler.fit_transform(X_train)
scaler.transform(X_test)
```

- 훈련 데이터에서 평균과 표준편차를 학습한다.
- 테스트 데이터에는 같은 기준으로 변환만 적용한다.
- 테스트 데이터에 `fit_transform`을 쓰면 데이터 누수 위험이 있다.

## 18. DataLoader 만들기

전체 데이터를 한 번에 넣지 않고 mini-batch로 나눠 학습한다.

### 함수 사용법: `TensorDataset`

```python
dataset = TensorDataset(X_tensor, y_tensor)
```

- 입력 Tensor와 정답 Tensor를 한 묶음으로 만든다.

### 함수 사용법: `DataLoader`

```python
DataLoader(dataset, batch_size=64, shuffle=True)
```

- 데이터를 batch 단위로 꺼낸다.
- `shuffle=True`는 매 epoch마다 데이터 순서를 섞는다.

In [ ]:
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print("train batch 개수:", len(train_loader))
print("test batch 개수:", len(test_loader))

> 필기 포인트:  
> Mini-batch 학습은 메모리를 아끼고, 업데이트를 더 자주 하게 만든다.  
> 1 epoch 안에서도 batch 수만큼 `optimizer.step()`이 실행된다.

## 19. He 초기화를 적용한 MLP 만들기

ReLU를 쓰는 네트워크에서는 He 초기화가 자주 사용된다.

### 함수 사용법: `nn.init.kaiming_normal_()`

```python
nn.init.kaiming_normal_(layer.weight)
```

- ReLU 계열 활성화 함수에 적합한 초기화다.
- `kaiming`은 He 초기화의 PyTorch 함수명이다.

### 함수 사용법: `nn.init.zeros_()`

```python
nn.init.zeros_(layer.bias)
```

- bias를 0으로 초기화한다.

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

        for m in self.f:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.f(x)

model = MLPClassifier().to(device)

print(model)

> 기억할 점:  
> 출력이 2개인 이유는 class 0과 class 1의 점수를 각각 내기 위해서다.  
> 이 경우 손실 함수는 `CrossEntropyLoss`를 사용한다.

## 20. Scheduler 준비하기

Learning Rate Scheduler는 학습률을 자동으로 조절한다.

강의 자료에서는 초반에는 크게 움직이고, 후반에는 작게 조정하는 전략을 설명했다.

### 함수 사용법: `OneCycleLR`

```python
optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,
    steps_per_epoch=len(train_loader),
    epochs=5
)
```

- `max_lr`: 가장 높은 학습률이다.
- `steps_per_epoch`: 한 epoch 안의 step 수다.
- `epochs`: 전체 epoch 수다.

> 필기 포인트:  
> Scheduler는 보폭을 자동으로 조절하는 도구다.

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=3e-3)

num_epochs = 5

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs
)

criterion_ce = nn.CrossEntropyLoss()

print("optimizer:", optimizer)
print("scheduler:", scheduler)
print("criterion:", criterion_ce)

### 함수 사용법: `nn.CrossEntropyLoss()`

```python
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
```

- `logits`: 모델의 raw score다. Softmax를 직접 붙이지 않는다.
- `labels`: 정답 class index다. 예: 0 또는 1
- 내부적으로 LogSoftmax와 NLLLoss가 함께 처리된다.

## 21. Scheduler와 Gradient 기록을 포함한 학습

이번에는 epoch마다 test accuracy와 gradient 평균을 기록한다.

In [ ]:
grad_values = []

def collect_grad_hook(grad):
    grad_values.append(grad.detach().abs().mean().item())
    return grad

hook_handles = []

for name, param in model.named_parameters():
    if "weight" in name:
        hook_handles.append(param.register_hook(collect_grad_hook))

hist_grad = []
hist_acc = []
hist_lr = []

for epoch in range(num_epochs):
    model.train()

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion_ce(outputs, batch_y)

        loss.backward()
        optimizer.step()
        scheduler.step()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_X)
            pred = outputs.argmax(1)

            correct += (pred == batch_y).sum().item()
            total += batch_y.size(0)

    recent_grad = np.mean(grad_values[-len(train_loader):]) if len(grad_values) > 0 else 0.0

    hist_grad.append(recent_grad)
    hist_acc.append(correct / total)
    hist_lr.append(optimizer.param_groups[0]["lr"])

    print(f"epoch {epoch + 1} | test_acc={hist_acc[-1]:.3f} | avg_grad={hist_grad[-1]:.6f} | lr={hist_lr[-1]:.6f}")

for handle in hook_handles:
    handle.remove()

코드 설명:

- `param.register_hook(...)`으로 weight gradient를 기록한다.
- `outputs.argmax(1)`로 각 샘플의 예측 class를 구한다.
- `scheduler.step()`은 batch마다 학습률을 갱신한다.
- `optimizer.param_groups[0]["lr"]`로 현재 learning rate를 확인한다.

> 헷갈림 포인트:  
> Scheduler는 Optimizer가 parameter를 업데이트하는 방식 자체를 바꾸는 것이 아니라, 주로 learning rate를 조절한다.

In [ ]:
plt.plot(hist_grad, label="average |grad|")
plt.xlabel("epoch")
plt.ylabel("gradient magnitude")
plt.title("Average Gradient Magnitude")
plt.legend()
plt.show()

In [ ]:
plt.plot(hist_acc, label="test accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Test Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.plot(hist_lr, label="learning rate")
plt.xlabel("epoch")
plt.ylabel("lr")
plt.title("Learning Rate History")
plt.legend()
plt.show()

그래프 해석:

- Gradient가 너무 0에 가까우면 학습 신호가 약할 수 있다.
- Accuracy가 올라가면 분류 성능이 좋아지고 있다는 뜻이다.
- Learning rate는 Scheduler 전략에 따라 변한다.

## 22. Batch Size가 학습에 미치는 영향

Batch size는 한 번에 몇 개 데이터를 보고 업데이트할지 정하는 값이다.

| Batch size | 특징 |
|---|---|
| 작음 | 노이즈가 많지만 일반화에 도움될 수 있다 |
| 큼 | gradient가 안정적이지만 메모리를 많이 쓴다 |

강의 자료에서는 보통 16~128 정도를 많이 선택한다고 정리했다.

In [ ]:
class SimpleBinaryNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

y_train_binary = torch.FloatTensor(y_train).unsqueeze(1)

batch_sizes = [8, 32, 128, 512]
criterion_bce = nn.BCELoss()

batch_size_results = []

for batch_size in batch_sizes:
    dataset = TensorDataset(X_train_t, y_train_binary)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model_bs = SimpleBinaryNet()
    optimizer_bs = optim.Adam(model_bs.parameters(), lr=0.001)

    losses = []

    for epoch in range(5):
        epoch_loss = 0.0

        for batch_X, batch_y in loader:
            model_bs.train()
            optimizer_bs.zero_grad()

            output = model_bs(batch_X)
            loss = criterion_bce(output, batch_y)

            loss.backward()
            optimizer_bs.step()

            epoch_loss += loss.item()

        losses.append(epoch_loss / len(loader))

    batch_size_results.append((batch_size, losses))

for batch_size, losses in batch_size_results:
    print(f"batch_size={batch_size} | final_loss={losses[-1]:.4f}")

In [ ]:
for batch_size, losses in batch_size_results:
    plt.plot(losses, label=f"Batch={batch_size}")

plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("Effect of Batch Size")
plt.legend()
plt.show()

그래프 해석:

- 작은 batch는 loss가 조금 흔들릴 수 있다.
- 큰 batch는 더 안정적일 수 있지만, 항상 일반화 성능이 좋은 것은 아니다.
- 실제 프로젝트에서는 메모리, 속도, 성능을 보고 batch size를 조정한다.

## 23. BatchNorm 없는 모델과 있는 모델 만들기

BatchNorm은 각 층의 입력 분포를 안정화하는 방법이다.

강의 자료에서는 Internal Covariate Shift, 즉 학습 중 각 층의 입력 분포가 계속 바뀌는 문제를 줄여준다고 설명했다.

### 함수 사용법: `nn.BatchNorm1d()`

```python
nn.BatchNorm1d(num_features)
```

- 1차원 feature 벡터를 정규화한다.
- `Linear` 출력이 `[batch, features]` 형태일 때 사용한다.

In [ ]:
class NetWithoutBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


class NetWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(20, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

print(NetWithoutBN())
print("\n--- With BN ---")
print(NetWithBN())

BatchNorm을 넣는 대표 순서는 다음이다.

```text
Linear → BatchNorm → ReLU
```

또는 경우에 따라:

```text
Linear → ReLU → BatchNorm
```

도 가능하지만, 기본적으로는 Linear 다음에 BatchNorm을 넣는 흐름을 먼저 기억하면 된다.

## 24. BatchNorm 효과 비교 학습

BatchNorm이 없는 모델과 있는 모델을 같은 데이터로 학습한다.

In [ ]:
def train_binary_model(model, loader, epochs=5, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    losses = []

    for epoch in range(epochs):
        epoch_loss = 0.0

        for batch_X, batch_y in loader:
            model.train()
            optimizer.zero_grad()

            output = model(batch_X)
            loss = criterion(output, batch_y)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        losses.append(epoch_loss / len(loader))

    return losses

dataset = TensorDataset(X_train_t, y_train_binary)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model_no_bn = NetWithoutBN()
model_bn = NetWithBN()

loss_no_bn = train_binary_model(model_no_bn, loader)
loss_bn = train_binary_model(model_bn, loader)

print("Without BN final loss:", loss_no_bn[-1])
print("With BN final loss:", loss_bn[-1])

In [ ]:
plt.plot(loss_no_bn, label="Without BN")
plt.plot(loss_bn, label="With BN")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("BatchNorm Effect")
plt.legend()
plt.show()

그래프 해석:

- BatchNorm이 있는 모델이 더 안정적으로 내려가면 학습 안정화 효과가 있다고 볼 수 있다.
- 데이터와 초기값에 따라 항상 극적으로 좋아지는 것은 아니지만, 깊은 네트워크에서 도움이 되는 경우가 많다.

## 25. BatchNorm과 Gradient 안정성 비교

BatchNorm은 gradient 흐름을 더 고르게 만들 수 있다.

Gradient를 Hook으로 기록해서 층별 gradient 크기를 비교한다.

In [ ]:
grad_hist_without_bn = []
grad_hist_with_bn = []

model_without_bn = NetWithoutBN()
model_with_bn = NetWithBN()

def create_gradient_hook(gradient_list):
    def hook(grad):
        gradient_list.append(grad.abs().mean().item())
        return grad
    return hook

handles_no_bn = []
handles_bn = []

for name, param in model_without_bn.named_parameters():
    if "weight" in name:
        handles_no_bn.append(param.register_hook(create_gradient_hook(grad_hist_without_bn)))

for name, param in model_with_bn.named_parameters():
    if "weight" in name:
        handles_bn.append(param.register_hook(create_gradient_hook(grad_hist_with_bn)))

batch_X = X_train_t[:32]
batch_y = y_train_binary[:32]

criterion = nn.BCELoss()

output_no_bn = model_without_bn(batch_X)
loss_no_bn_once = criterion(output_no_bn, batch_y)
loss_no_bn_once.backward()

output_bn = model_with_bn(batch_X)
loss_bn_once = criterion(output_bn, batch_y)
loss_bn_once.backward()

for h in handles_no_bn + handles_bn:
    h.remove()

print("Without BN gradients:")
print(grad_hist_without_bn)

print("\nWith BN gradients:")
print(grad_hist_with_bn)

In [ ]:
plt.bar(range(1, len(grad_hist_without_bn) + 1), grad_hist_without_bn)
plt.xlabel("layer number")
plt.ylabel("gradient magnitude")
plt.title("Gradient Magnitude without BatchNorm")
plt.show()

In [ ]:
plt.bar(range(1, len(grad_hist_with_bn) + 1), grad_hist_with_bn)
plt.xlabel("layer number")
plt.ylabel("gradient magnitude")
plt.title("Gradient Magnitude with BatchNorm")
plt.show()

그래프 해석:

- 특정 층의 gradient가 너무 작으면 그 층은 거의 학습되지 않을 수 있다.
- 특정 층의 gradient가 너무 크면 학습이 불안정해질 수 있다.
- BatchNorm은 층별 gradient 흐름을 조금 더 안정적으로 만드는 데 도움을 줄 수 있다.

## 26. BatchNorm의 train/eval 모드 차이

BatchNorm은 학습 모드와 평가 모드에서 동작이 다르다.

```text
model.train(): 현재 mini-batch의 평균/분산을 사용한다.
model.eval(): 학습 중 누적된 running mean/variance를 사용한다.
```

그래서 BatchNorm이 있는 모델은 `train()`과 `eval()` 구분이 특히 중요하다.

In [ ]:
model_bn_mode = NetWithBN()
optimizer_mode = optim.Adam(model_bn_mode.parameters(), lr=0.001)

loader_mode = DataLoader(dataset, batch_size=64, shuffle=True)

for epoch in range(5):
    for batch_X, batch_y in loader_mode:
        model_bn_mode.train()

        optimizer_mode.zero_grad()
        output = model_bn_mode(batch_X)
        loss = criterion_bce(output, batch_y)

        loss.backward()
        optimizer_mode.step()

sample_test = X_test_t[:10]

model_bn_mode.train()
with torch.no_grad():
    pred_train_mode = model_bn_mode(sample_test)

model_bn_mode.eval()
with torch.no_grad():
    pred_eval_mode = model_bn_mode(sample_test)

print("train mode prediction mean:", pred_train_mode.mean().item())
print("eval mode prediction mean:", pred_eval_mode.mean().item())

> 기억할 점:  
> 평가나 예측할 때 `model.eval()`을 빼먹으면 BatchNorm이나 Dropout 때문에 결과가 흔들릴 수 있다.

## 27. Xavier 초기화와 He 초기화

강의 자료에서는 초기화 전략도 정리했다.

| 초기화 | 주로 어울리는 활성화 함수 |
|---|---|
| Xavier / Glorot | Tanh, Sigmoid |
| He / Kaiming | ReLU, LeakyReLU |

### 함수 사용법

```python
nn.init.xavier_normal_(layer.weight)
nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
nn.init.orthogonal_(layer.weight)
```

- Xavier는 Tanh/Sigmoid 계열에 자주 사용한다.
- He는 ReLU 계열에 자주 사용한다.
- Orthogonal은 RNN/LSTM/GRU에서 자주 언급된다.

In [ ]:
layer_xavier = nn.Linear(20, 64)
layer_he = nn.Linear(20, 64)

nn.init.xavier_normal_(layer_xavier.weight)
nn.init.kaiming_normal_(layer_he.weight, nonlinearity="relu")

print("Xavier weight std:", layer_xavier.weight.std().item())
print("He weight std:", layer_he.weight.std().item())

> 필기 포인트:  
> 초기화가 너무 작으면 신호가 사라지고, 너무 크면 폭발할 수 있다.  
> 그래서 활성화 함수에 맞는 초기화를 선택하는 것이 중요하다.

## 28. BatchNorm과 LayerNorm 구분

강의 마지막에서는 BatchNorm과 LayerNorm을 구분했다.

| 정규화 | 기준 | 자주 쓰는 곳 |
|---|---|---|
| BatchNorm | batch 차원의 통계 | CNN, MLP |
| LayerNorm | 각 샘플의 feature 차원 통계 | RNN, Transformer |

### 함수 사용법

```python
nn.BatchNorm1d(num_features)
nn.LayerNorm(normalized_shape)
```

- BatchNorm은 batch 크기 영향을 받는다.
- LayerNorm은 각 샘플 내부 feature를 기준으로 정규화한다.

In [ ]:
bn = nn.BatchNorm1d(10)
ln = nn.LayerNorm(10)

sample = torch.randn(4, 10)

bn_out = bn(sample)
ln_out = ln(sample)

print("sample shape:", sample.shape)
print("BatchNorm output shape:", bn_out.shape)
print("LayerNorm output shape:", ln_out.shape)

> 기억할 점:  
> Transformer 계열에서는 보통 BatchNorm보다 LayerNorm을 더 자주 본다.  
> 시퀀스 길이나 batch 구성이 변해도 각 샘플 단위로 안정적으로 처리하기 좋기 때문이다.

## 29. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `net` | 예측 함수 | 입력을 받아 예측값을 출력한다 |
| `criterion` | 손실 함수 | 예측값과 정답을 비교한다 |
| `optimizer` | 최적화 함수 | 파라미터를 업데이트한다 |
| `Layer` | Tensor 변환 함수 | `nn.Linear`, `nn.ReLU` |
| `Parameter` | 학습되는 값 | weight, bias |
| `nn.Linear` | 선형 Layer | `nn.Linear(in, out)` |
| `nn.ReLU` | 활성화 함수 | 음수는 0, 양수는 그대로 |
| `nn.Sequential` | Layer 묶음 | 순서대로 실행되는 모델 |
| `forward` | 순전파 | 입력에서 출력까지 계산 |
| `backward` | 역전파 | 손실에서 gradient 계산 |
| `loss` | 손실 | 예측이 얼마나 틀렸는지 |
| `grad` | gradient | 파라미터 수정 방향 |
| `argmax` | 최댓값 위치 | `out.argmax(1)` |
| `item` | Python 숫자 추출 | `loss.item()` |
| `DataLoader` | mini-batch 생성 | `DataLoader(dataset, batch_size=...)` |
| `TensorDataset` | Tensor 묶음 | 입력과 정답을 묶는다 |
| `Batch Size` | 한 번에 학습할 데이터 수 | 32, 64, 128 등 |
| `Scheduler` | 학습률 조절기 | `OneCycleLR`, `StepLR` |
| `Hook` | gradient 관찰 도구 | `register_hook()` |
| `BatchNorm` | batch 기준 정규화 | `nn.BatchNorm1d()` |
| `LayerNorm` | 샘플 feature 기준 정규화 | `nn.LayerNorm()` |
| `He 초기화` | ReLU용 초기화 | `kaiming_normal_()` |
| `Xavier 초기화` | Tanh/Sigmoid용 초기화 | `xavier_normal_()` |

## 30. 시험용 요약

```text
PyTorch 모델 = Layer 함수들을 순서대로 연결한 예측 함수
```

꼭 기억할 것:

- Layer 함수는 Tensor를 입력받아 Tensor를 출력한다.
- Parameter는 Layer 내부에서 학습되는 weight와 bias다.
- Model은 Layer 함수들을 조합한 큰 합성 함수다.
- `nn.Linear(in, out)`에서 앞 Layer의 out과 다음 Layer의 in이 맞아야 한다.
- `nn.Sequential()`은 Layer들을 순서대로 묶는다.
- PyTorch 학습의 3대 요소는 `net`, `criterion`, `optimizer`다.
- 학습 4단계는 예측, 손실, 경사, 업데이트다.
- `optimizer.zero_grad()`는 gradient 초기화다.
- `loss.backward()`는 역전파다.
- `optimizer.step()`은 파라미터 업데이트다.
- 활성화 함수가 없으면 Linear를 여러 번 쌓아도 결국 선형이다.
- ReLU가 있으면 비선형 패턴을 학습할 수 있다.
- `argmax(1)`은 각 행에서 가장 큰 class index를 반환한다.
- `item()`은 원소 하나짜리 Tensor를 Python 숫자로 꺼낸다.
- Hook은 gradient 흐름을 관찰하는 도구다.
- Scheduler는 learning rate를 조절한다.
- He 초기화는 ReLU 계열에 적합하다.
- Xavier 초기화는 Tanh/Sigmoid 계열에 적합하다.
- Mini-batch는 메모리와 학습 효율을 위해 데이터를 작은 묶음으로 나누는 방식이다.
- BatchNorm은 각 층의 입력 분포를 안정화한다.
- BatchNorm은 gradient 흐름과 학습 안정성에 도움을 줄 수 있다.
- BatchNorm이 있는 모델은 `model.train()`과 `model.eval()` 구분이 중요하다.
- LayerNorm은 RNN, Transformer 계열에서 자주 쓰인다.